<a href="https://colab.research.google.com/github/matiaxx/santander-dev-week-2023/blob/master/SantanderDevWeek2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚨 A API Está Fora do Ar? Sem Problemas!

A API utilizada neste projeto (`sdw-2023-prd.up.railway.app`) era um ambiente público de demonstração e foi descontinuada. **Isso não impede você de concluir este desafio** 🚀

Como reforçado nas aulas, o objetivo principal aqui é compreender o fluxo **ETL (Extração, Transformação e Carregamento)**. Se a API não estiver respondendo, você pode substituir a etapa de **Extract** e ajustar a **Load** sem prejuízo ao aprendizado.


## Ajustando a Etapa de Extract

### 🟢 Opção 1: Mais Simples (Dados Direto no Código)

Ideal se você quiser focar exclusivamente na lógica e no uso de IA, sem depender de arquivos externos.

```python
# Simula a extração de dados (substituindo o GET da API)
users = [
    {'id': 1, 'name': 'Naruto', 'news': []},
    {'id': 2, 'name': 'Hinata', 'news': []}
]
```

### 🟡 Opção 2: Mais Completa (Leitura de Arquivo)

Você pode adaptar o arquivo `SDW2023.csv`, incluindo a coluna `name`, e usá-lo como fonte de dados.

```python
import pandas as pd

# Lê o CSV e converte para uma lista de dicionários
users = pd.read_csv('SDW2023.csv').to_dict(orient='records')

# Garante a estrutura esperada para a etapa de Transformação
for user in users:
    user['news'] = []
```

## E a Etapa de Load?

Como a API está indisponível, a chamada `PUT` não funcionará. Para finalizar o Lab, basta salvar o resultado localmente (em um arquivo CSV, JSON ou mesmo exibindo no console).

O mais importante é demonstrar que você entende como os dados percorrem todas as etapas do **ETL**, independentemente da ferramenta ou fonte utilizada 😉

# Santander Dev Week 2023 (ETL com Python)

**Contexto:** Você é um cientista de dados no Santander e recebeu a tarefa de envolver seus clientes de maneira mais personalizada. Seu objetivo é usar o poder da IA Generativa para criar mensagens de marketing personalizadas que serão entregues a cada cliente.

**Condições do Problema:**

1. Você recebeu uma planilha simples, em formato CSV ('SDW2023.csv'), com uma lista de IDs de usuário do banco:
  ```
  UserID
  1
  2
  3
  4
  5
  ```
2. Seu trabalho é consumir o endpoint `GET https://sdw-2023-prd.up.railway.app/users/{id}` (API da Santander Dev Week 2023) para obter os dados de cada cliente.
3. Depois de obter os dados dos clientes, você vai usar a API do ChatGPT (OpenAI) para gerar uma mensagem de marketing personalizada para cada cliente. Essa mensagem deve enfatizar a importância dos investimentos.
4. Uma vez que a mensagem para cada cliente esteja pronta, você vai enviar essas informações de volta para a API, atualizando a lista de "news" de cada usuário usando o endpoint `PUT https://sdw-2023-prd.up.railway.app/users/{id}`.



In [ ]:
# Utilize sua própria URL se quiser ;)
# Repositório da API: https://github.com/digitalinnovationone/santander-dev-week-2023-api
sdw2023_api_url = 'https://sdw-2023-prd.up.railway.app'

## **E**xtract

Extraia a lista de IDs de usuário a partir do arquivo CSV. Para cada ID, faça uma requisição GET para obter os dados do usuário correspondente.

In [1]:
import pandas as pd

df = pd.read_csv('SDW2023.csv')
user_ids = df['UserID'].tolist()
print(user_ids)

[4, 5, 6]


## **T**ransform

Utilize a API do OpenAI GPT-4 para gerar uma mensagem de marketing personalizada para cada usuário.

In [6]:
pip install -q google-generativeai

In [12]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY2')
genai.configure(api_key=GOOGLE_API_KEY)

In [19]:
# Initialize the Gemini API
# Check for available models if 'gemini-pro' is not working
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

gemini_model = genai.GenerativeModel('gemini-2.5-flash') # Using 'gemini-pro-latest' from the list above

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [20]:
# Atualizando a função para usar o Gemini API
def generate_ai_news_gemini(user):
  # Define as instruções para o modelo e o conteúdo do usuário
  prompt = f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)"

  # Gera o conteúdo usando o modelo Gemini
  response = gemini_model.generate_content(prompt)

  # Retorna o texto gerado, removendo aspas extras se houver
  return response.text.strip('"')

# Prepara a lista de usuários a partir do DataFrame, garantindo a estrutura esperada
# Renomeia 'UserID' para 'id' e inicializa 'news' como uma lista vazia para cada usuário.
users = df.to_dict(orient='records')
for user in users:
    # Renomear 'UserID' para 'id' para consistência com o restante do projeto
    user['id'] = user.pop('UserID')
    user['news'] = [] # Inicializa a lista 'news' para cada usuário

# Loop para gerar notícias para cada usuário com o Gemini API
for user in users:
  news_description = generate_ai_news_gemini(user)
  print(f"Notícia para {user['name']}: {news_description}")
  user['news'].append({
      "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
      "description": news_description
  })

Notícia para Pyterson: Pyterson, invista! Seu futuro financeiro depende de você. Comece hoje!
Notícia para Pip: Pip, invista! Seu dinheiro cresce e garante um futuro tranquilo. Comece hoje! (72 caracteres)
Notícia para Pep: Pep, investimento contínuo: o segredo para evoluir e manter o nível de campeão.


## **L**oad

Atualize a lista de "news" de cada usuário na API com a nova mensagem gerada.

In [21]:
import pandas as pd

# Converte a lista de dicionários 'users' para um DataFrame
users_df = pd.DataFrame(users)

# Expande a coluna 'news' para que cada notícia seja uma linha separada, se necessário
# Ou, para simplicidade, se houver apenas uma notícia por usuário, pode-se acessá-la diretamente
# Neste caso, vamos assumir que queremos a descrição da primeira notícia, ou um campo JSON da lista de notícias

# Para salvar a coluna 'news' como uma string JSON, o que preserva a estrutura da lista de dicionários:
users_df['news'] = users_df['news'].apply(lambda x: str(x))

# Salva o DataFrame em um novo arquivo CSV
output_csv_filename = 'SDW2023_updated.csv'
users_df.to_csv(output_csv_filename, index=False)

print(f"Arquivo '{output_csv_filename}' gerado com sucesso!")

Arquivo 'SDW2023_updated.csv' gerado com sucesso!
